# Session 3: Linear Algebra Operations (2 hours)

**Goal:** Implement **one step of linear regression** (predictions + loss) so shapes and matrix–vector product become intuitive. Then implement matvec from scratch and verify with NumPy.

| Section | Time | Focus |
|---------|------|-------|
| Anchor case 1 | 35 min | One LR step: predictions, loss, gradient shape |
| Anchor case 2 | 30 min | Batch of weight vectors (X @ W.T → (n, k)) |
| Anchor case 3 | 30 min | Numerical gradient check (analytical vs finite diff) |
| Task 3.1 | 45 min | Matrix operations from scratch (matvec, matmul, outer, transpose) |
| Task 3.2 | 40 min | Decompositions |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

---
## Anchor case: One step of linear regression [MEDIUM]

**Task:** Given X (n×d), w (d,), b (scalar), compute **predictions** = X @ w + b (shape (n,)), then **loss** = mean((y - predictions)²). This is the forward pass and loss we'll use in Session 4–5.

**Shapes:** X (n, d) @ w (d,) → (n,); add b (scalar) → (n,). y is (n,). So (y - predictions) is (n,), squared and mean → scalar.

### Worked example

In [ ]:
np.random.seed(42)
n, d = 20, 3
X = np.random.randn(n, d)   # (n, d)
w = np.random.randn(d)     # (d,)
b = 0.5
y = np.random.randn(n)    # (n,) targets

predictions = X @ w + b    # (n, d) @ (d,) + scalar → (n,)
loss = np.mean((y - predictions) ** 2)
print(f"X {X.shape}, w {w.shape} → predictions {predictions.shape}")
print(f"Loss (MSE): {loss:.4f}")

### Your turn [MEDIUM]

1. Same formulas with **n=50, d=4**. Compute predictions and loss.
2. The gradient of MSE w.r.t. w is: **dw = (2/n) * X.T @ (predictions - y)**. Compute `dw` and check its shape is (d,).

In [ ]:
np.random.seed(0)
n, d = 50, 4
X = np.random.randn(n, d)
w = np.random.randn(d)
b = 0.0
y = np.random.randn(n)

predictions = ...   # YOUR CODE (n,)
loss = ...          # YOUR CODE scalar
dw = ...            # (2/n) * X.T @ (predictions - y)

print(f"predictions.shape = {predictions.shape}, loss = {loss:.4f}")
print(f"dw.shape = {dw.shape}")
assert dw.shape == (d,), f"Expected (d,), got {dw.shape}"

<details>
<summary>💡 Click to see solution</summary>

```python
predictions = X @ w + b
loss = np.mean((y - predictions) ** 2)
dw = (2 / n) * (X.T @ (predictions - y))
```
</details>

### Sensemaking

**What is the shape of the gradient of the loss w.r.t. w? How did you check you were right?**

_Your answer: _

---
## Anchor case 2: Batch of weight vectors (one matrix multiply) [MEDIUM]

**Task:** We have X of shape (n, d) (n examples, d features) and **W** of shape (k, d) — k different weight vectors (e.g. k classifiers). We want predictions for **all** n examples under **all** k weight vectors in one go. Result shape (n, k): out[i, j] = X[i] @ W[j].

**Hint:** (n, d) @ (d, k) = (n, k). So we need X @ W.T (since W is (k, d), W.T is (d, k)).

### Worked example

In [ ]:
n, d, k = 5, 3, 2
X = np.random.randn(n, d)
W = np.random.randn(k, d)   # k weight vectors, each of length d
# (n, d) @ (d, k) = (n, k)
all_preds = X @ W.T
print(f"X {X.shape}, W {W.shape} -> all_preds {all_preds.shape}")
print("Row 0 predictions for all k:", all_preds[0])
print("Manual check X[0] @ W[0]:", X[0] @ W[0])

### Your turn [MEDIUM]

Given X (20, 4) and W (5, 4), compute the (20, 5) matrix of predictions. Then verify one row: all_preds[3] should equal X[3] @ W.T.

In [ ]:
np.random.seed(0)
X = np.random.randn(20, 4)
W = np.random.randn(5, 4)
all_preds = ...  # YOUR CODE (20, 5)
print(all_preds.shape)
assert np.allclose(all_preds[3], X[3] @ W.T)

<details>
<summary>💡 Click to see solution</summary>

```python
all_preds = X @ W.T
```
</details>

---
## Anchor case 3: Numerical gradient check [MEDIUM]

**Task:** Verify that your analytical gradient **dw** is correct by comparing it to a **numerical** gradient. For one weight w_j, the numerical derivative is: (loss(w + h*e_j) - loss(w - h*e_j)) / (2*h). Compute numerical dw for a small random X, y, w and compare to your analytical dw (element-wise). They should match up to ~1e-5.

### Worked example

In [ ]:
def mse_loss(X, w, b, y):
    return np.mean((X @ w + b - y) ** 2)

def grad_w_analytical(X, w, b, y):
    n = len(y)
    pred = X @ w + b
    return (2 / n) * (X.T @ (pred - y))

def grad_w_numerical(X, w, b, y, h=1e-5):
    dw = np.zeros_like(w)
    for j in range(len(w)):
        e = np.zeros_like(w)
        e[j] = 1
        dw[j] = (mse_loss(X, w + h*e, b, y) - mse_loss(X, w - h*e, b, y)) / (2 * h)
    return dw

np.random.seed(42)
n, d = 10, 3
X = np.random.randn(n, d)
w = np.random.randn(d)
b = 0.0
y = np.random.randn(n)
dw_ana = grad_w_analytical(X, w, b, y)
dw_num = grad_w_numerical(X, w, b, y)
print("Analytical:", dw_ana)
print("Numerical: ", dw_num)
print("Max diff:", np.abs(dw_ana - dw_num).max())

### Your turn [MEDIUM]

Run the numerical check above (or your own) for a different random seed. Confirm max difference is small (< 1e-4).

### Sensemaking

**Why is the numerical gradient useful when you implement a new loss or model?**

_Your answer: _

---
## Task 3.1: Matrix operations from scratch (45 min)

Implement **matvec** with explicit loops to build intuition; then verify with NumPy. Same idea as the anchor case: each row of A dotted with v.

### 3.1a: Matrix-Vector Multiplication

In [ ]:
def matvec_manual(A, v):
    """
    Compute A @ v manually.
    A: (m, n) matrix
    v: (n,) vector
    Returns: (m,) vector
    """
    m, n = A.shape
    assert v.shape[0] == n, f"Dimension mismatch: A is {A.shape}, v is {v.shape}"

    result = np.zeros(m)
    for i in range(m):
        for j in range(n):
            pass  # YOUR CODE HERE


### 3.1b: Matrix-Matrix Multiplication

In [ ]:
def matmul_manual(A, B):
    """
    Compute A @ B manually.
    A: (m, n) matrix
    B: (n, p) matrix
    Returns: (m, p) matrix
    """
    m, n = A.shape
    n2, p = B.shape
    assert n == n2, f"Inner dimensions must match: {n} != {n2}"

    result = np.zeros((m, p))
    # TODO: Fill in the triple nested loop
    for i in range(m):
        for j in range(p):
            for k in range(n):
                pass  # YOUR CODE HERE


### 3.1c: Outer Product

In [ ]:
def outer_product_manual(a, b):
    """
    Compute outer product of vectors a and b.
    a: (m,) vector
    b: (n,) vector
    Returns: (m, n) matrix where result[i,j] = a[i] * b[j]
    """
    m = a.shape[0]
    n = b.shape[0]
    result = np.zeros((m, n))

    # TODO: Fill in the loop
    for i in range(m):
        for j in range(n):
            pass  # YOUR CODE HERE


### 3.1d: Matrix Transpose (from scratch)

In [ ]:
def transpose_manual(A):
    """
    Transpose a matrix.
    A: (m, n) matrix
    Returns: (n, m) matrix
    """
    m, n = A.shape
    result = np.zeros((n, m))

    # TODO: Fill in the loop
    for i in range(m):
        for j in range(n):
            pass  # YOUR CODE HERE


### 3.1e: Speed Comparison

How much faster is NumPy's `@` compared to your manual implementation?

In [ ]:
np.random.seed(42)
size = 100
A = np.random.randn(size, size)
B = np.random.randn(size, size)

print(f"Matrix size: {size}x{size}")

%timeit matmul_manual(A, B)
%timeit A @ B

print("\nNumPy uses optimized BLAS libraries written in C/Fortran!")

---
## Task 3.2: Decompositions (1 hour)

### 3.2a: Eigenvalue Decomposition

For a square matrix A, the eigendecomposition finds vectors **v** and scalars **lambda** such that:

A @ v = lambda * v

Geometrically: eigenvectors are directions that only get scaled (not rotated) by the transformation A.

In [ ]:
A = np.array([[4, 2],
              [1, 3]])

eigenvalues, eigenvectors = np.linalg.eig(A)

print(f"Eigenvalues: {eigenvalues}")
print(f"Eigenvectors (columns):\n{eigenvectors}")

# Verify: A @ v should equal lambda * v for each eigenpair
for i in range(len(eigenvalues)):
    v = eigenvectors[:, i]
    lam = eigenvalues[i]
    lhs = A @ v
    rhs = lam * v
    print(f"\nEigenpair {i}: lambda = {lam:.4f}")
    print(f"  A @ v  = {lhs}")
    print(f"  lam*v  = {rhs}")
    print(f"  Match: {np.allclose(lhs, rhs)}")

### 3.2b: Visualize Eigenvectors

See how the matrix transforms regular vectors versus eigenvectors.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot unit circle and its transformation
theta = np.linspace(0, 2 * np.pi, 100)
circle = np.array([np.cos(theta), np.sin(theta)])
transformed = A @ circle

for ax, data, title in [(axes[0], circle, 'Before: Unit Circle'),
                         (axes[1], transformed, f'After: A @ circle')]:
    ax.plot(data[0], data[1], 'b-', linewidth=2)

    # Draw eigenvectors
    for i in range(len(eigenvalues)):
        v = eigenvectors[:, i]
        if title.startswith('After'):
            v = eigenvalues[i] * v
        ax.annotate('', xy=v, xytext=(0, 0),
                     arrowprops=dict(arrowstyle='->', color='red', lw=2))
        ax.annotate(f'v{i+1} (λ={eigenvalues[i]:.1f})',
                     xy=v, fontsize=10, color='red')

    ax.set_xlim(-6, 6)
    ax.set_ylim(-6, 6)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)
    ax.set_title(title, fontsize=14)

plt.tight_layout()
plt.show()

### 3.2c: Singular Value Decomposition (SVD)

SVD decomposes **any** matrix A (not just square) into: A = U @ S @ V^T

- **U**: left singular vectors (rotation)
- **S**: singular values (scaling)
- **V^T**: right singular vectors (rotation)

In [ ]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])

U, s, Vt = np.linalg.svd(A)

print(f"A shape: {A.shape}")
print(f"U shape: {U.shape}")
print(f"s (singular values): {s}")
print(f"Vt shape: {Vt.shape}")

# Reconstruct A from SVD
S = np.zeros_like(A, dtype=float)
S[:min(A.shape), :min(A.shape)] = np.diag(s)

A_reconstructed = U @ S @ Vt
print(f"\nReconstructed A:\n{A_reconstructed}")
print(f"Match: {np.allclose(A, A_reconstructed)}")

### 3.2d: Solving Linear Systems

Solve Ax = b without computing A^(-1) (which is numerically unstable).

In [ ]:
# System: 2x + y = 5, x + 3y = 7
A = np.array([[2, 1],
              [1, 3]])
b = np.array([5, 7])

# Method 1: np.linalg.solve (preferred)
x_solve = np.linalg.solve(A, b)

# Method 2: Using inverse (avoid in practice)
x_inv = np.linalg.inv(A) @ b

print(f"Solution (solve): x = {x_solve}")
print(f"Solution (inv):   x = {x_inv}")

# Verify: A @ x should equal b
print(f"\nA @ x = {A @ x_solve} (should be {b})")
print(f"Match: {np.allclose(A @ x_solve, b)}")

### 3.2e: Key Properties to Verify

Check your understanding by verifying these properties.

In [ ]:
np.random.seed(42)
A = np.random.randn(3, 3)
B = np.random.randn(3, 3)

# Property 1: (AB)^T = B^T A^T
prop1 = np.allclose((A @ B).T, B.T @ A.T)
print(f"(AB)^T = B^T A^T: {prop1}")

# Property 2: (A^T)^T = A
prop2 = np.allclose(A.T.T, A)
print(f"(A^T)^T = A: {prop2}")

# Property 3: det(AB) = det(A) * det(B)
prop3 = np.allclose(np.linalg.det(A @ B),
                     np.linalg.det(A) * np.linalg.det(B))
print(f"det(AB) = det(A)*det(B): {prop3}")

# Property 4: trace(A) = sum of eigenvalues
eigenvalues = np.linalg.eigvals(A)
prop4 = np.allclose(np.trace(A), np.sum(eigenvalues))
print(f"trace(A) = sum(eigenvalues): {prop4}")

# Property 5: det(A) = product of eigenvalues
prop5 = np.allclose(np.linalg.det(A), np.prod(eigenvalues))
print(f"det(A) = prod(eigenvalues): {prop5}")

### Checkpoint

Can you explain what eigenvectors represent geometrically?

**Your answer (double-click to edit):**

_Write your explanation here..._

**Connection:** The anchor case (predictions = X @ w + b, loss, dw) is exactly what we use in Session 4 for gradient descent and in Session 5 for the full linear regression class.

---
## Session 3 Reflection

**How much slower was your manual matmul vs NumPy?**

_Your answer..._

**Why is `np.linalg.solve` preferred over computing the inverse?**

_Your answer..._

**What are singular values and how do they relate to eigenvalues?**

_Your answer..._